In [0]:
import requests

openf1_stints_url = "https://api.openf1.org/v1/stints"
openf1_drivers_url = "https://api.openf1.org/v1/drivers"

stints_response = requests.get(
    openf1_stints_url,
    params={"session_key": "latest"},
    timeout=30
)

drivers_response = requests.get(
    openf1_drivers_url,
    params={"session_key": "latest"},
    timeout=30
)

stints_response.raise_for_status()
drivers_response.raise_for_status()

openf1_stints = stints_response.json()
openf1_drivers = drivers_response.json()

print("Stint records:", len(openf1_stints))
print("Driver records:", len(openf1_drivers))

In [0]:
print("Example stint:")
print(openf1_stints[0])

print("\nExample driver:")
print(openf1_drivers[0])

In [0]:
catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

silver_results_table = f"{catalog}.f1_silver.race_results"

silver_results_df = spark.table(silver_results_table)

driver_mapping_df = (
    silver_results_df
    .select(
        "race_id",
        "driver_id",
        "driver_code"
    )
    .dropDuplicates()
)

display(driver_mapping_df)

In [0]:
openf1_stints_with_codes_df = (
    openf1_stints_df
    .join(
        openf1_drivers_df.select(
            "driver_number",
            "name_acronym"
        ),
        on="driver_number",
        how="left"
    )
)

display(
    openf1_stints_with_codes_df.select(
        "driver_number",
        "name_acronym",
        "compound",
        "lap_start",
        "lap_end",
        "stint_number"
    )
)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

stints_schema = StructType([
    StructField("compound", StringType(), True),
    StructField("driver_number", IntegerType(), True),
    StructField("lap_end", IntegerType(), True),
    StructField("lap_start", IntegerType(), True),
    StructField("meeting_key", IntegerType(), True),
    StructField("session_key", IntegerType(), True),
    StructField("stint_number", IntegerType(), True),
    StructField("tyre_age_at_start", IntegerType(), True)
])

drivers_schema = StructType([
    StructField("driver_number", IntegerType(), True),
    StructField("name_acronym", StringType(), True)
])

openf1_stints_df = spark.createDataFrame(
    openf1_stints,
    stints_schema
)

openf1_drivers_df = spark.createDataFrame(
    openf1_drivers,
    drivers_schema
)

display(openf1_stints_df)
display(openf1_drivers_df)

In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

silver_results_table = f"{catalog}.f1_silver.race_results"

silver_results_df = spark.table(silver_results_table)

driver_mapping_df = (
    silver_results_df
    .select("driver_id", "driver_code")
    .dropDuplicates()
)

In [0]:
stints_with_codes_df = (
    openf1_stints_df
    .join(
        openf1_drivers_df.select(
            "driver_number",
            "name_acronym"
        ),
        on="driver_number",
        how="left"
    )
)

compound_stints_df = (
    stints_with_codes_df
    .join(
        driver_mapping_df,
        F.upper(
            stints_with_codes_df["name_acronym"]
        ) == F.upper(
            driver_mapping_df["driver_code"]
        ),
        how="left"
    )
    .select(
        "driver_id",
        "driver_code",
        "driver_number",
        "stint_number",
        "lap_start",
        "lap_end",
        "compound",
        "tyre_age_at_start"
    )
)

display(compound_stints_df)

In [0]:
display(
    compound_stints_df
    .filter(F.col("driver_id").isNull())
)

In [0]:
import requests
from pyspark.sql import functions as F

sessions_response = requests.get(
    "https://api.openf1.org/v1/sessions",
    params={"session_key": "latest"},
    timeout=30
)

sessions_response.raise_for_status()

latest_sessions = sessions_response.json()

print("OpenF1 latest session:")
print(latest_sessions[:1])

In [0]:
silver_results_table = f"{catalog}.f1_silver.race_results"

current_race = (
    spark.table(silver_results_table)
    .select(
        "race_id",
        "race_name",
        "season",
        "round"
    )
    .dropDuplicates()
    .first()
)

print("Jolpica race:")
print(current_race)

In [0]:
import requests

current_race = (
    spark.table(silver_results_table)
    .select(
        "race_id",
        "race_name",
        "season",
        "round"
    )
    .dropDuplicates()
    .first()
)

target_year = int(current_race["season"])
target_race_name = current_race["race_name"]

print("Target Jolpica race:", target_race_name)
print("Target season:", target_year)

In [0]:
meetings_response = requests.get(
    "https://api.openf1.org/v1/meetings",
    params={"year": target_year},
    timeout=30
)

meetings_response.raise_for_status()

meetings = meetings_response.json()

for meeting in meetings:
    print(
        meeting.get("meeting_key"),
        meeting.get("meeting_name"),
        meeting.get("date_start")
    )

In [0]:
matching_meeting_key = 1296 

In [0]:
race_sessions_response = requests.get(
    "https://api.openf1.org/v1/sessions",
    params={
        "meeting_key": matching_meeting_key,
        "session_name": "Race"
    },
    timeout=30
)

race_sessions_response.raise_for_status()

race_sessions = race_sessions_response.json()

print(race_sessions)

In [0]:
matching_session_key = race_sessions[0]["session_key"]

print("Matching OpenF1 session:", matching_session_key)

In [0]:
stints_response = requests.get(
    "https://api.openf1.org/v1/stints",
    params={"session_key": matching_session_key},
    timeout=30
)

stints_response.raise_for_status()

openf1_stints = stints_response.json()

print("Matching stint records:", len(openf1_stints))
print(openf1_stints[:2])

In [0]:
drivers_response = requests.get(
    "https://api.openf1.org/v1/drivers",
    params={"session_key": matching_session_key},
    timeout=30
)

drivers_response.raise_for_status()

openf1_drivers = drivers_response.json()

print("Drivers found:", len(openf1_drivers))
print(openf1_drivers[:1])

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

openf1_stints_df = spark.createDataFrame(openf1_stints)

drivers_schema = StructType([
    StructField("driver_number", IntegerType(), True),
    StructField("name_acronym", StringType(), True),
])
openf1_drivers_df = spark.createDataFrame(openf1_drivers, schema=drivers_schema)

display(openf1_stints_df)
display(openf1_drivers_df)

In [0]:
stints_with_codes_df = (
    openf1_stints_df
    .join(
        openf1_drivers_df.select(
            "driver_number",
            "name_acronym"
        ),
        on="driver_number",
        how="left"
    )
)

In [0]:
driver_mapping_df = (
    spark.table(silver_results_table)
    .select(
        "driver_id",
        "driver_code"
    )
    .dropDuplicates()
)

In [0]:
compound_stints_df = (
    stints_with_codes_df
    .join(
        driver_mapping_df,
        F.upper(
            stints_with_codes_df["name_acronym"]
        ) == F.upper(
            driver_mapping_df["driver_code"]
        ),
        how="left"
    )
    .select(
        F.lit(current_race["race_id"]).alias("race_id"),
        "driver_id",
        F.col("name_acronym").alias("driver_code"),
        "driver_number",
        "stint_number",
        F.col("lap_start").alias("stint_start_lap"),
        F.col("lap_end").alias("stint_end_lap"),
        F.upper("compound").alias("tyre_compound"),
        "tyre_age_at_start"
    )
)

display(compound_stints_df)

In [0]:
display(
    compound_stints_df
    .filter(F.col("driver_id").isNull())
)